# YouTube Video Transcript Readability Analysis

This notebook demonstrates the data preprocessing pipeline for analyzing YouTube video transcripts using text readability features.

## Overview
- **Goal**: Build features to classify YouTube videos as "understandable" or not
- **Data Structure**: Video metadata + caption_text (transcripts)
- **Features**: 
    1. Text readability metrics including Flesch Reading Ease, grade levels, and lexical diversity


In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

# Import our custom feature engineering module
import sys
import os
sys.path.append(os.path.dirname(os.path.abspath('.')))
from feature_engineering import (
    calculate_readability_features,
    batch_calculate_readability_features,
    flesch_reading_ease,
    flesch_kincaid_grade_level,
    smog_index,
    gunning_fog_index,
    average_word_length,
    long_word_ratio,
    lexical_diversity
)

print("Libraries imported successfully!")
print("Feature engineering module loaded!")


## 1. Data Import (Placeholder)

Since the actual YouTube video data is not ready yet, we'll create placeholder data to demonstrate the feature calculation process.

**Expected Data Structure:**
- `video_id`: Unique identifier for each video
- `title`: Video title
- `description`: Video description
- `caption_text`: Full transcript/caption text
- `duration`: Video duration
- `view_count`: Number of views
- `like_count`: Number of likes
- `category`: Video category
- `upload_date`: When the video was uploaded


In [ ]:
# Create placeholder data for demonstration
np.random.seed(42)

# Sample video data with different complexity levels
sample_data = {
    'video_id': [f'video_{i:03d}' for i in range(1, 11)],
    'title': [
        'Simple Health Tips for Everyone',
        'Advanced Medical Procedures Explained',
        'Basic Nutrition Guide',
        'Complex Surgical Techniques',
        'Easy Exercise Routines',
        'Sophisticated Research Findings',
        'Simple Cooking Recipes',
        'Advanced Pharmacology Concepts',
        'Basic First Aid Instructions',
        'Complex Diagnostic Methods'
    ],
    'description': [
        'Learn simple health tips that anyone can follow.',
        'Detailed explanation of advanced medical procedures.',
        'Basic nutrition information for beginners.',
        'In-depth analysis of complex surgical techniques.',
        'Easy exercises you can do at home.',
        'Comprehensive research findings and analysis.',
        'Simple recipes for healthy cooking.',
        'Advanced concepts in pharmacology.',
        'Basic first aid for common injuries.',
        'Complex diagnostic methods and procedures.'
    ],
    'caption_text': [
        # Simple text (high readability)
        "Hello everyone. Today we will talk about simple health tips. First, drink plenty of water. Second, eat fresh fruits and vegetables. Third, get enough sleep. These are easy steps anyone can follow. They will help you stay healthy and feel good.",
        
        # Complex text (low readability)
        "The implementation of sophisticated pharmacological interventions necessitates comprehensive understanding of molecular mechanisms underlying therapeutic efficacy. Contemporary research methodologies employ advanced analytical techniques to elucidate complex biochemical pathways and their intricate interactions within cellular microenvironments.",
        
        # Medium complexity
        "Nutrition is important for your health. You should eat a balanced diet with different types of food. Include vegetables, fruits, proteins, and whole grains. Avoid too much sugar and processed foods. This will help your body work properly.",
        
        # Very complex text
        "Surgical procedures require meticulous attention to anatomical structures and physiological considerations. The implementation of minimally invasive techniques necessitates sophisticated instrumentation and advanced imaging modalities to ensure optimal patient outcomes while minimizing postoperative complications.",
        
        # Simple text
        "Exercise is good for you. Start with simple movements. Walk for 30 minutes each day. Do some stretching. Lift light weights. These activities will make you stronger and healthier.",
        
        # Complex research text
        "Empirical investigations utilizing randomized controlled trials demonstrate statistically significant correlations between therapeutic interventions and clinical outcomes. Meta-analytical approaches reveal substantial effect sizes across diverse patient populations, suggesting robust generalizability of treatment protocols.",
        
        # Simple cooking text
        "Today I will show you how to make a simple salad. First, wash your vegetables. Then cut them into small pieces. Add some olive oil and salt. Mix everything together. Your healthy salad is ready to eat.",
        
        # Complex medical text
        "Pharmacokinetic parameters including bioavailability, distribution volume, and elimination half-life significantly influence therapeutic drug monitoring protocols. Dose optimization strategies must account for individual patient characteristics and potential drug interactions.",
        
        # Simple first aid
        "If someone gets hurt, stay calm. First, check if they are breathing. If not, call for help immediately. Apply pressure to stop bleeding. Keep the person warm and comfortable until help arrives.",
        
        # Complex diagnostic text
        "Diagnostic algorithms incorporating machine learning methodologies enable sophisticated pattern recognition within multidimensional datasets. Advanced computational approaches facilitate identification of subtle pathological indicators through comprehensive analysis of clinical parameters and imaging biomarkers."
    ],
    'duration': np.random.randint(60, 1800, 10),  # 1-30 minutes
    'view_count': np.random.randint(1000, 1000000, 10),
    'like_count': np.random.randint(50, 50000, 10),
    'category': np.random.choice(['Health', 'Education', 'Science', 'Lifestyle'], 10),
    'upload_date': pd.date_range('2023-01-01', periods=10, freq='D')
}

# Create DataFrame
df = pd.DataFrame(sample_data)

print("Sample data created successfully!")
print(f"Data shape: {df.shape}")
print("\nFirst few rows:")
df.head()


## 2. Feature Calculation

Now we'll calculate the readability features for each video transcript using our feature engineering module.


In [ ]:
# Calculate readability features for all videos
print("Calculating readability features...")

# Initialize lists to store features
readability_features = []

# Calculate features for each video
for idx, row in df.iterrows():
    print(f"Processing video {idx + 1}/10: {row['title']}")
    
    # Get the caption text
    caption_text = row['caption_text']
    
    # Calculate all readability features
    features = calculate_readability_features(caption_text)
    
    # Add video metadata to features
    features['video_id'] = row['video_id']
    features['title'] = row['title']
    features['category'] = row['category']
    
    readability_features.append(features)

# Convert to DataFrame
features_df = pd.DataFrame(readability_features)

print(f"\nFeatures calculated successfully!")
print(f"Features shape: {features_df.shape}")
print("\nFeature columns:", list(features_df.columns))


In [ ]:
# Display the calculated features
print("Readability Features Results:")
print("=" * 50)

# Show all features with proper formatting
feature_columns = [
    'flesch_reading_ease', 'flesch_kincaid_grade_level', 'smog_index', 
    'gunning_fog_index', 'average_word_length', 'long_word_ratio', 'lexical_diversity'
]

for col in feature_columns:
    if col in features_df.columns:
        print(f"\n{col.upper()}:")
        print(f"  Mean: {features_df[col].mean():.3f}")
        print(f"  Std:  {features_df[col].std():.3f}")
        print(f"  Min:  {features_df[col].min():.3f}")
        print(f"  Max:  {features_df[col].max():.3f}")

# Show detailed results
print("\nDetailed Results:")
features_df[['video_id', 'title'] + feature_columns].round(3)


## 3. Feature Analysis and Visualization


In [ ]:
# Create visualizations for readability features
plt.style.use('default')
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Readability Features Analysis', fontsize=16, fontweight='bold')

# Plot each feature
feature_names = [
    'flesch_reading_ease', 'flesch_kincaid_grade_level', 'smog_index', 
    'gunning_fog_index', 'average_word_length', 'long_word_ratio', 'lexical_diversity'
]

feature_labels = [
    'Flesch Reading Ease', 'Flesch-Kincaid Grade', 'SMOG Index', 
    'Gunning Fog Index', 'Avg Word Length', 'Long Word Ratio', 'Lexical Diversity'
]

for i, (feature, label) in enumerate(zip(feature_names, feature_labels)):
    row = i // 4
    col = i % 4
    
    if i < len(feature_names):
        axes[row, col].bar(range(len(features_df)), features_df[feature], 
                          color=plt.cm.viridis(features_df[feature] / features_df[feature].max()))
        axes[row, col].set_title(label, fontweight='bold')
        axes[row, col].set_xlabel('Video Index')
        axes[row, col].set_ylabel('Score')
        axes[row, col].tick_params(axis='x', rotation=45)
        
        # Add value labels on bars
        for j, v in enumerate(features_df[feature]):
            axes[row, col].text(j, v + 0.01, f'{v:.2f}', ha='center', va='bottom', fontsize=8)

# Remove empty subplot
if len(feature_names) < 8:
    axes[1, 3].remove()

plt.tight_layout()
plt.show()


In [ ]:
# Correlation matrix of readability features
plt.figure(figsize=(10, 8))

# Select only numeric feature columns
numeric_features = features_df[feature_names]
correlation_matrix = numeric_features.corr()

# Create heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
plt.title('Readability Features Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print correlation insights
print("Key Correlations:")
print("=" * 30)
for i in range(len(feature_names)):
    for j in range(i+1, len(feature_names)):
        corr = correlation_matrix.iloc[i, j]
        if abs(corr) > 0.5:  # Strong correlation
            print(f"{feature_names[i]} vs {feature_names[j]}: {corr:.3f}")


## 4. Feature Interpretation and Insights


In [ ]:
# Interpret readability scores
print("READABILITY SCORE INTERPRETATION")
print("=" * 50)

# Flesch Reading Ease interpretation
print("\n1. FLESCH READING EASE (0-100 scale):")
print("   90-100: Very Easy (5th grade)")
print("   80-89:  Easy (6th grade)")
print("   70-79:  Fairly Easy (7th grade)")
print("   60-69:  Standard (8th-9th grade)")
print("   50-59:  Fairly Difficult (10th-12th grade)")
print("   30-49:  Difficult (College level)")
print("   0-29:   Very Difficult (Graduate level)")

# Grade level interpretation
print("\n2. GRADE LEVEL SCORES:")
print("   Flesch-Kincaid: U.S. grade level needed")
print("   SMOG Index: Years of education needed")
print("   Gunning Fog: Years of formal education needed")

# Other metrics
print("\n3. OTHER METRICS:")
print("   Average Word Length: Syllables per word (lower = simpler)")
print("   Long Word Ratio: Proportion of 3+ syllable words (lower = simpler)")
print("   Lexical Diversity: Vocabulary richness (0-1, higher = more diverse)")

# Analyze our sample data
print("\n" + "="*50)
print("SAMPLE DATA ANALYSIS")
print("="*50)

# Find easiest and hardest videos
easiest_idx = features_df['flesch_reading_ease'].idxmax()
hardest_idx = features_df['flesch_reading_ease'].idxmin()

print(f"\nEasiest Video: {features_df.loc[easiest_idx, 'title']}")
print(f"Flesch Score: {features_df.loc[easiest_idx, 'flesch_reading_ease']:.1f}")
print(f"Grade Level: {features_df.loc[easiest_idx, 'flesch_kincaid_grade_level']:.1f}")

print(f"\nHardest Video: {features_df.loc[hardest_idx, 'title']}")
print(f"Flesch Score: {features_df.loc[hardest_idx, 'flesch_reading_ease']:.1f}")
print(f"Grade Level: {features_df.loc[hardest_idx, 'flesch_kincaid_grade_level']:.1f}")

# Overall statistics
print(f"\nOverall Statistics:")
print(f"Average Flesch Score: {features_df['flesch_reading_ease'].mean():.1f}")
print(f"Average Grade Level: {features_df['flesch_kincaid_grade_level'].mean():.1f}")
print(f"Average Lexical Diversity: {features_df['lexical_diversity'].mean():.3f}")
